# 04 — Detektimi i përjashtuesve

**Autor:** Dafina Keqmezi
**Versioni:** 1.0
**Data:** 18.11.2026
**Statusi:** Pjesa II. Deri më 29.11.2026 punohet mbi `processed/biodiversity_features.csv`, pastaj mbi `processed/biodiversity_final.csv`.

Ky notebook vetëm **shënon** përjashtuesit. Nuk fshin dhe nuk ndryshon asnjë vlerë. Vendimi nëse një përjashtues është gabim merret te `05_menjanimi.ipynb`.

## 1. Plani i detektimit të përjashtuesve (18 nëntor)

Për çdo kolonë kandidate shkruhet metoda dhe dallimi ndërmjet dy rasteve:
- **gabimi i mundshëm**: vlera që nuk mund të jetë e vërtetë;
- **vlera reale por e rrallë**: vlera ekstreme por e besueshme, që nuk duhet fshirë.

Numrat në tabelë llogariten më poshtë nga të dhënat, jo me dorë.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = ROOT / "processed"
REPORTS_DIR = ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(FIGURES_DIR / ".gitkeep").touch()

FEATURES_PATH = PROCESSED_DIR / "biodiversity_features.csv"
AGG_PATH = PROCESSED_DIR / "agg_municipality_v2.csv"

df = pd.read_csv(FEATURES_PATH, low_memory=False)
agg = pd.read_csv(AGG_PATH)
assert df.shape[0] == 46_456 and df["gbifID"].is_unique
assert agg.shape[0] == 38 and agg["komuna"].is_unique
assert agg["regjistrime"].sum() == len(df)

ROW_COLUMNS = ["individualCount", "elevation", "coordinateUncertaintyInMeters",
               "year", "decimalLatitude", "decimalLongitude", "n_issues"]
AGG_COLUMNS = ["regjistrime", "specie_unike", "specie_per_km2", "regjistrime_per_1000_banore",
               "specie_te_kercenuara", "regjistrime_te_kercenuara"]
LOG_COLUMNS = {"individualCount", "coordinateUncertaintyInMeters", *AGG_COLUMNS}

# Kufijtë nga Pjesa I (reports/korniza_e_kualitetit.md, reports/vlefshmeria.md).
ELEV_MIN, ELEV_MAX = 297, 2_656          # Drini i Bardhë, Gjeravica
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 41.85, 43.27, 20.01, 21.80
print(df.shape, agg.shape)

(46456, 128) (38, 13)


### 1.1 Pamja e parë e kolonave kandidate

`IQR 1.5` tregon sa vlera do të shënonte IQR me k = 1.5 në shkallë lineare dhe, për kolonat e anuara, mbi `log1p`. Kjo është vetëm orientim për zgjedhjen e metodës. Metoda IQR zbatohet më 20 nëntor.

In [2]:
def iqr_bounds(values, k=1.5):
    q1, q3 = values.quantile([0.25, 0.75])
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)

def iqr_count(values, k=1.5):
    low, high = iqr_bounds(values, k)
    return int(((values < low) | (values > high)).sum())

def profile(frame, columns, source):
    rows = []
    for column in columns:
        values = frame[column].dropna()
        rows.append({
            "burimi": source, "kolona": column,
            "n": len(values), "mungon_pct": round(frame[column].isna().mean() * 100, 2),
            "min": values.min(), "q1": values.quantile(0.25), "mediana": values.median(),
            "q3": values.quantile(0.75), "max": values.max(),
            "moda": values.mode().iloc[0], "moda_n": int(values.eq(values.mode().iloc[0]).sum()),
            "iqr15_lineare": iqr_count(values),
            "iqr15_log1p": iqr_count(np.log1p(values)) if column in LOG_COLUMNS else np.nan,
        })
    return pd.DataFrame(rows)

candidate_profile = pd.concat([profile(df, ROW_COLUMNS, "rreshta"),
                               profile(agg, AGG_COLUMNS, "komuna")], ignore_index=True)
candidate_profile.to_csv(REPORTS_DIR / "04_kolonat_kandidate.csv", index=False)
display(candidate_profile)

,burimi,kolona,n,mungon_pct,min,q1,mediana,q3,max,moda,moda_n,iqr15_lineare,iqr15_log1p
0,rreshta,individualCount,28869,37.86,1.000000,1.000000,2.000000,6.000000,1.020000e+04,1.000000,10992,3628,1093.0
1,rreshta,elevation,6168,86.72,100.000000,498.000000,665.135000,1064.637500,3.050000e+03,481.000000,749,595,NaN
2,rreshta,coordinateUncertaintyInMeters,14643,68.48,1.000000,250.000000,7071.000000,7071.000000,4.226661e+06,7071.000000,8225,66,1.0
3,rreshta,year,44188,4.88,1826.000000,2018.000000,2020.000000,2022.000000,2.026000e+03,2020.000000,8370,6180,NaN
4,rreshta,decimalLatitude,46456,0.00,41.858976,42.561417,42.579243,42.659138,4.324167e+01,42.579243,12943,11476,NaN
5,rreshta,decimalLongitude,46456,0.00,20.020786,20.815317,21.048304,21.143663,2.176290e+01,21.048304,12943,4338,NaN
6,rreshta,n_issues,46456,0.00,0.000000,0.000000,0.000000,0.000000,3.000000e+00,0.000000,44811,1645,NaN
7,komuna,regjistrime,38,0.00,1.000000,181.500000,343.000000,960.750000,1.456300e+04,24.000000,2,6,3.0
8,komuna,specie_unike,38,0.00,1.000000,114.000000,189.000000,316.750000,1.243000e+03,114.000000,2,6,4.0
9,komuna,specie_per_km2,38,0.00,0.091000,0.376250,0.797000,1.861000,3.368000e+00,0.335000,2,0,0.0


### 1.2 Faktet që përdoren në plan

In [3]:
f = {"N": len(df)}

# individualCount
ic = df["individualCount"].dropna()
big = df[df["individualCount"] >= 1_000]
f |= {"ic_n": len(ic), "ic_med": ic.median(), "ic_max": ic.max(),
      "ic_big": len(big), "ic_big_sturnus": int(big["species"].eq("Sturnus vulgaris").sum()),
      "ic_big_grac": int(big["municipality"].str.startswith("Graçanicë").sum()),
      "ic_big_clo": int(big["institutionCode"].eq("CLO").sum()),
      "ic_10000": int(df["individualCount"].between(10_000, 10_200).sum()),
      "ic_flag": int(df["flag_INDIVIDUAL_COUNT_INVALID"].sum()),
      "ic_flag_nan": int(df.loc[df["flag_INDIVIDUAL_COUNT_INVALID"].eq(1), "individualCount"].isna().sum()),
      "ic_insect_max": df.loc[df["taxon_group"].eq("Insekte"), "individualCount"].max(),
      "ic_iqr_lin": iqr_count(ic), "ic_iqr_log": iqr_count(np.log1p(ic))}
assert f["ic_flag"] == f["ic_flag_nan"]

# elevation
el = df["elevation"].dropna()
high = df[df["elevation"] > ELEV_MAX]
low = df[df["elevation"] < ELEV_MIN]
f |= {"el_n": len(el), "el_zero": int(df["elevation_was_zero"].sum()),
      "el_hi": len(high), "el_hi_min": high["elevation"].min(), "el_hi_max": high["elevation"].max(),
      "el_hi_inst": high["institutionCode"].unique().tolist(), "el_hi_year": high["year"].unique().tolist(),
      "el_hi_points": high[["decimalLatitude", "decimalLongitude"]].drop_duplicates().shape[0],
      "el_lo": len(low), "el_lo_values": sorted(low["elevation"].tolist()),
      "el_iqr_lin": iqr_count(el), "el_iqr_high": iqr_bounds(el)[1],
      "el_fung_med": df.loc[df["taxon_group"].eq("Kërpudha"), "elevation"].median(),
      "el_ins_med": df.loc[df["taxon_group"].eq("Insekte"), "elevation"].median()}
assert f["el_hi_points"] == 1 and f["el_hi_inst"] == ["WU"]

# coordinateUncertaintyInMeters
cu = df["coordinateUncertaintyInMeters"].dropna()
huge = df[df["coordinateUncertaintyInMeters"] > 100_000]
mid = df[df["coordinateUncertaintyInMeters"].between(10_000, 100_000, inclusive="right")]
f |= {"cu_n": len(cu), "cu_7071": int(cu.eq(7_071).sum()), "cu_7071_pct": cu.eq(7_071).mean() * 100,
      "cu_huge": len(huge), "cu_huge_min": huge["coordinateUncertaintyInMeters"].min(),
      "cu_max": cu.max(), "cu_huge_inat": int(huge["institutionCode"].eq("iNaturalist").sum()),
      "cu_mid": len(mid), "cu_mid_inat": int(mid["institutionCode"].eq("iNaturalist").sum()),
      "cu_le5": int(cu.le(5).sum()),
      "cu_iqr_lin": iqr_count(cu), "cu_iqr_log": iqr_count(np.log1p(cu))}
assert f["cu_huge"] == f["cu_huge_inat"]

# year
yr = df["year"].dropna()
old = df[df["year"] < 1900]
f |= {"yr_miss": int(df["year"].isna().sum()), "yr_min": int(yr.min()), "yr_max": int(yr.max()),
      "yr_last_date": pd.to_datetime(df["event_start"], errors="coerce").max().date(),
      "yr_pre1900": len(old), "yr_pre1900_ps": int(old["basisOfRecord"].eq("PRESERVED_SPECIMEN").sum()),
      "yr_boom_pct": yr.between(2018, 2022).mean() * 100,
      "yr_iqr_lin": iqr_count(yr), "yr_iqr_low": iqr_bounds(yr)[0]}

# decimalLatitude / decimalLongitude
lat, lon = df["decimalLatitude"], df["decimalLongitude"]
points = df.groupby(["decimalLatitude", "decimalLongitude"]).size().sort_values(ascending=False)
two_dec = np.isclose(lat, lat.round(2)) & np.isclose(lon, lon.round(2))
f |= {"ll_out": int((~lat.between(LAT_MIN, LAT_MAX) | ~lon.between(LON_MIN, LON_MAX)).sum()),
      "ll_lat": (lat.min(), lat.max()), "ll_lon": (lon.min(), lon.max()),
      "ll_points": len(points), "ll_top": points.index[0], "ll_top_n": int(points.iloc[0]),
      "ll_2dec": int(two_dec.sum())}

# n_issues
ni = df["n_issues"]
f |= {"ni_counts": ni.value_counts().sort_index().to_dict(), "ni_ge1": int(ni.ge(1).sum()),
      "ni_ge2": int(ni.ge(2).sum()), "ni_3": int(ni.eq(3).sum()), "ni_iqr_lin": iqr_count(ni)}
assert f["ni_iqr_lin"] == f["ni_ge1"]   # Q1 = Q3 = 0: IQR shënon çdo flamur

# agregatet e komunave
def extreme(column, n=2, largest=True):
    s = agg.set_index("komuna")[column]
    s = s.nlargest(n) if largest else s.nsmallest(n)
    return ", ".join(f"{k} {v:,.2f}".replace(".00", "") for k, v in s.items())

f |= {f"agg_{c}_iqr_log": iqr_count(np.log1p(agg[c])) for c in AGG_COLUMNS}
f |= {"agg_grac_pct": agg.set_index("komuna").loc["Graçanicë", "perqindja_e_regjistrimeve"],
      "agg_est": agg.loc[agg["popullsia_vleresim_ask"], "komuna"].tolist(),
      "agg_shared_area": agg.loc[agg["njesia_e_siperfaqes"].str.contains(r"\+"), "komuna"].tolist(),
      "agg_no_threat": int(agg["specie_te_kercenuara"].eq(0).sum())}
display(pd.Series(f, name="vlera").to_frame())

,vlera
N,46456
ic_n,28869
ic_med,2.0
ic_max,10200.0
ic_big,79
...,...
agg_regjistrime_te_kercenuara_iqr_log,2
agg_grac_pct,31.35
agg_est,"[Leposaviq, Zveçan, Zubin Potok, Mitrovicë e V..."
agg_shared_area,"[Mitrovicë e Jugut, Mitrovicë e Veriut]"


### 1.3 Tabela e planit

In [4]:
n = lambda x: f"{x:,.0f}"
ROWS = [
 {"Kolona": "`individualCount`",
  "Metoda": f"IQR me k = 1.5 dhe 3 mbi `log1p`, sepse shpërndarja është shumë e anuar (mediana {n(f['ic_med'])}, maksimumi {n(f['ic_max'])}; IQR lineare shënon {n(f['ic_iqr_lin'])} vlera, mbi `log1p` {n(f['ic_iqr_log'])}). Z-score i modifikuar (MAD, \\|Mz\\| > 3.5). IQR brenda `taxon_group` dhe specieve. Isolation Forest / LOF me `log1p`. Z-score klasik vetëm për krahasim.",
  "Gabim i mundshëm": f"Shifër e tepërt ose gabim shtypi (p.sh. 102 → 10,200). Numri i tufës i shkruar te çdo specie e listës. Numër i futur si kod. GBIF shënoi `INDIVIDUAL_COUNT_INVALID` te {n(f['ic_flag'])} rreshta. Te të gjithë vlera mungon tashmë, prandaj nuk hyjnë në detektim.",
  "Vlerë reale e rrallë": f"Tufa dimërore shturësh: {n(f['ic_big'])} rreshta kanë ≥ 1,000 individë, {n(f['ic_big_sturnus'])} prej tyre *Sturnus vulgaris*, {n(f['ic_big_grac'])} në Graçanicë dhe {n(f['ic_big_clo'])} nga eBird. {n(f['ic_10000'])} vlera janë 10,000–10,200: vlerësim i rrumbullakët i vëzhguesit, jo numërim. Te insektet deri në {n(f['ic_insect_max'])} individë."},
 {"Kolona": "`elevation`",
  "Metoda": f"Rregulla fizike {n(ELEV_MIN)}–{n(ELEV_MAX)} m nga Pjesa I. IQR dhe MAD brenda `taxon_group` dhe `class` (përjashtues kontekstual). Isolation Forest / LOF bashkë me koordinatat. IQR globale nuk mjafton: shënon {n(f['el_iqr_lin'])} vlera mbi {n(f['el_iqr_high'])} m, kryesisht regjistrime malore.",
  "Gabim i mundshëm": f"Mbi Gjeravicën ({n(ELEV_MAX)} m): {f['el_hi']} ekzemplarë të herbariumit WU nga viti {', '.join(str(int(y)) for y in f['el_hi_year'])}, në të njëjtën pikë në Kaçanik, me {n(f['el_hi_min'])}–{n(f['el_hi_max'])} m. Nën {ELEV_MIN} m: {f['el_lo']} vlera ({', '.join(n(v) for v in f['el_lo_values'])} m). Lartësi nga etiketa e vjetër, në këmbë në vend të metrave, ose që nuk i përket koordinatës. {n(f['el_zero'])} zerot u kthyen në NaN në Pjesën I.",
  "Vlerë reale e rrallë": f"Kërpudhat dhe likenet kanë medianë {n(f['el_fung_med'])} m, insektet {n(f['el_ins_med'])} m: regjistrohen kryesisht në male (te PCA e Pjesës I, Dragashi dhe Juniku kanë pjesën më të madhe të kërpudhave). Vlerat {n(f['el_iqr_high'])}–{n(ELEV_MAX)} m janë të rralla për bimët dhe insektet, por janë brenda relievit të Kosovës."},
 {"Kolona": "`coordinateUncertaintyInMeters`",
  "Metoda": f"IQR mbi `log1p` (lineare shënon {n(f['cu_iqr_lin'])} vlera, `log1p` {n(f['cu_iqr_log'])}). MAD. Krahasim me pragun hapësinor të komunës nga Pjesa I (1,866 m). Moda 7,071 m trajtohet si kod i përgjithësimit, jo si përjashtues.",
  "Gabim i mundshëm": f"{f['cu_huge']} vlera mbi 100 km ({n(f['cu_huge_min'])}–{n(f['cu_max'])} m), më të mëdha se vetë Kosova. Të gjitha janë nga iNaturalist: ka gjasë gabim njësie ose vlerë e pajisjes. Pasiguri më e madhe se komuna e shënuar.",
  "Vlerë reale e rrallë": f"7,071 m ({n(f['cu_7071'])} rreshta, {f['cu_7071_pct']:.1f}% e të plotësuarave) është gjysma e diagonales së katrorit 10 × 10 km: koordinatë e përgjithësuar me qëllim. 10–100 km: {n(f['cu_mid'])} rreshta, {n(f['cu_mid_inat'])} nga iNaturalist, ndoshta koordinata të fshehura për specie të ndjeshme. ≤ 5 m ({n(f['cu_le5'])} rreshta): GPS i saktë."},
 {"Kolona": "`year`",
  "Metoda": f"Jo IQR ose Z global: vitet 2018–2022 kanë {f['yr_boom_pct']:.1f}% të regjistrimeve, prandaj IQR shënon çdo vit para {f['yr_iqr_low']:.0f} ({n(f['yr_iqr_lin'])} rreshta). IQR / MAD brenda `basisOfRecord`. Kontroll kundrejt `event_start` dhe `date_precision`. Për numrin e regjistrimeve sipas vitit/muajit: rolling median ± k·MAD te `agg_time_v2.csv`.",
  "Gabim i mundshëm": f"Vit pas datës së shkarkimit (regjistrimi i fundit: {f['yr_last_date']}). Viti i digjitalizimit në vend të vitit të mbledhjes. Vit që kundërshton `event_start` ose `date_precision`. {n(f['yr_miss'])} rreshta nuk kanë vit.",
  "Vlerë reale e rrallë": f"{f['yr_pre1900']} regjistrime para vitit 1900 (më i vjetri {f['yr_min']}), {f['yr_pre1900_ps']} prej tyre ekzemplarë herbariumi (`PRESERVED_SPECIMEN`). Janë dëshmi historike. Rritja pas 2018 vjen nga përpjekja e mbledhjes (eBird), jo nga ndryshim real."},
 {"Kolona": "`decimalLatitude` / `decimalLongitude`",
  "Metoda": f"Kontroll i kutisë së Kosovës (lat {LAT_MIN}–{LAT_MAX}, lon {LON_MIN:.2f}–{LON_MAX:.2f}). Distanca haversine deri te k = 5 fqinjët më të afërt, DBSCAN. Isolation Forest / LOF. IQR veç e veç mbi lat dhe lon nuk i gjen pikat e izoluara.",
  "Gabim i mundshëm": f"Pikë jashtë Kosovës ({f['ll_out']} tani: lat {f['ll_lat'][0]:.3f}–{f['ll_lat'][1]:.3f}, lon {f['ll_lon'][0]:.3f}–{f['ll_lon'][1]:.3f}). Lat dhe lon të ndërruara. Koordinata me ≤ 2 shifra dhjetore ({n(f['ll_2dec'])} rreshta), qendër komune ose rrjeti. Pikë që nuk përputhet me `municipality`.",
  "Vlerë reale e rrallë": f"Pika e Ligatinave të Henc ({f['ll_top'][0]}, {f['ll_top'][1]}) ka {n(f['ll_top_n'])} rreshta ({f['ll_top_n'] / f['N'] * 100:.1f}%): hotspot real, por i mbipërfaqësuar. Pikat e izoluara në male ose afër kufirit janë zona pak të vizituara, jo gabime."},
 {"Kolona": "`n_issues`",
  "Metoda": f"Nuk detektohet statistikisht: vlerat janë diskrete 0–3 dhe Q1 = Q3 = 0, prandaj IQR do të shënonte çdo rresht me ≥ 1 flamur ({n(f['ni_ge1'])}). Përdoret si variabël konteksti: krahasohet me `n_metoda` te flamujt.",
  "Gabim i mundshëm": f"Numri vetë nuk është matje e gabuar. Gabimi është te kolona që e ka flamurin (data, taksoni, pasiguria). {n(f['ni_ge2'])} rreshta me ≥ 2 flamuj kanë gjasë më të madhe gabimi.",
  "Vlerë reale e rrallë": f"{f['ni_3']} rreshta me 3 flamuj janë të rrallë, por GBIF i ka pranuar. Flamujt informues, p.sh. `TAXON_MATCH_HIGHERRANK`, nuk e bëjnë regjistrimin të gabuar."},
 {"Kolona": "`regjistrime` (komuna)",
  "Metoda": f"Mahalanobis mbi 38 komunat (regjistrime, specie unike, specie/km², regjistrime/1,000 banorë, % e kërcënuara), pas `log1p` dhe Z-score, me prag χ² (p = 0.975). IQR mbi `log1p` si krahasim ({f['agg_regjistrime_iqr_log']} komuna).",
  "Gabim i mundshëm": "Regjistrime në komunë të gabuar ose të numëruara dy herë. Duplikatet u hoqën në v2, prandaj mbetet rasti i komunës së gabuar.",
  "Vlerë reale e rrallë": f"Graçanicë {n(agg.set_index('komuna').loc['Graçanicë', 'regjistrime'])} regjistrime ({f['agg_grac_pct']:.2f}%), nga pika e Henc-it. Mamushë 1 regjistrim: komunë e vogël dhe pak e vizituar."},
 {"Kolona": "`specie_unike` (komuna)",
  "Metoda": f"Mahalanobis si më lart. IQR mbi `log1p` ({f['agg_specie_unike_iqr_log']} komuna).",
  "Gabim i mundshëm": "Sinonime ose emra në trajtime taksonomike të ndryshme që numërohen si specie të ndryshme (p.sh. *Lasiommata* / *Pararge* nga Pjesa I).",
  "Vlerë reale e rrallë": f"Më shumë: {extreme('specie_unike')}. Më pak: {extreme('specie_unike', largest=False)}. Prizreni ka Sharrin. Komunat e vogla kanë pak specie sepse kanë pak regjistrime."},
 {"Kolona": "`specie_per_km2` (komuna)",
  "Metoda": f"Mahalanobis si më lart. IQR mbi `log1p` ({f['agg_specie_per_km2_iqr_log']} komuna).",
  "Gabim i mundshëm": f"Sipërfaqe e gabuar ose e përbashkët: {' dhe '.join(f['agg_shared_area'])} kanë një njësi sipërfaqeje të përbashkët.",
  "Vlerë reale e rrallë": f"Më shumë: {extreme('specie_per_km2')}. Sipërfaqe e vogël dhe vende me shumë specie."},
 {"Kolona": "`regjistrime_per_1000_banore` (komuna)",
  "Metoda": f"Mahalanobis si më lart. IQR mbi `log1p` ({f['agg_regjistrime_per_1000_banore_iqr_log']} komuna).",
  "Gabim i mundshëm": f"Popullsia është vlerësim i ASK për {len(f['agg_est'])} komuna ({', '.join(f['agg_est'])}), prandaj raporti është më pak i sigurt aty.",
  "Vlerë reale e rrallë": f"Më shumë: {extreme('regjistrime_per_1000_banore')}. Përpjekje e lartë vëzhgimi në komuna me pak banorë."},
 {"Kolona": "`specie_te_kercenuara` / `regjistrime_te_kercenuara` (komuna)",
  "Metoda": f"Mahalanobis si më lart, me % e regjistrimeve të kërcënuara. IQR mbi `log1p` ({f['agg_specie_te_kercenuara_iqr_log']} dhe {f['agg_regjistrime_te_kercenuara_iqr_log']} komuna).",
  "Gabim i mundshëm": "Kategoria IUCN është globale, jo për Kosovën. Një identifikim i gabuar i një specieje të kërcënuar e rrit numrin.",
  "Vlerë reale e rrallë": f"Regjistrime: {extreme('regjistrime_te_kercenuara')}. Specie: {extreme('specie_te_kercenuara')}. {f['agg_no_threat']} komuna nuk kanë asnjë."},
]
plan = pd.DataFrame(ROWS)

def to_markdown(frame):
    lines = ["| " + " | ".join(frame.columns) + " |", "|" + "---|" * frame.shape[1]]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in frame.itertuples(index=False)]
    return "\n".join(lines)

plan_table = to_markdown(plan)
display(Markdown(plan_table))

| Kolona | Metoda | Gabim i mundshëm | Vlerë reale e rrallë |
|---|---|---|---|
| `individualCount` | IQR me k = 1.5 dhe 3 mbi `log1p`, sepse shpërndarja është shumë e anuar (mediana 2, maksimumi 10,200; IQR lineare shënon 3,628 vlera, mbi `log1p` 1,093). Z-score i modifikuar (MAD, \|Mz\| > 3.5). IQR brenda `taxon_group` dhe specieve. Isolation Forest / LOF me `log1p`. Z-score klasik vetëm për krahasim. | Shifër e tepërt ose gabim shtypi (p.sh. 102 → 10,200). Numri i tufës i shkruar te çdo specie e listës. Numër i futur si kod. GBIF shënoi `INDIVIDUAL_COUNT_INVALID` te 351 rreshta. Te të gjithë vlera mungon tashmë, prandaj nuk hyjnë në detektim. | Tufa dimërore shturësh: 79 rreshta kanë ≥ 1,000 individë, 73 prej tyre *Sturnus vulgaris*, 70 në Graçanicë dhe 78 nga eBird. 23 vlera janë 10,000–10,200: vlerësim i rrumbullakët i vëzhguesit, jo numërim. Te insektet deri në 600 individë. |
| `elevation` | Rregulla fizike 297–2,656 m nga Pjesa I. IQR dhe MAD brenda `taxon_group` dhe `class` (përjashtues kontekstual). Isolation Forest / LOF bashkë me koordinatat. IQR globale nuk mjafton: shënon 595 vlera mbi 1,915 m, kryesisht regjistrime malore. | Mbi Gjeravicën (2,656 m): 5 ekzemplarë të herbariumit WU nga viti 1890, në të njëjtën pikë në Kaçanik, me 2,750–3,050 m. Nën 297 m: 4 vlera (100, 198, 250, 293 m). Lartësi nga etiketa e vjetër, në këmbë në vend të metrave, ose që nuk i përket koordinatës. 3,634 zerot u kthyen në NaN në Pjesën I. | Kërpudhat dhe likenet kanë medianë 2,173 m, insektet 605 m: regjistrohen kryesisht në male (te PCA e Pjesës I, Dragashi dhe Juniku kanë pjesën më të madhe të kërpudhave). Vlerat 1,915–2,656 m janë të rralla për bimët dhe insektet, por janë brenda relievit të Kosovës. |
| `coordinateUncertaintyInMeters` | IQR mbi `log1p` (lineare shënon 66 vlera, `log1p` 1). MAD. Krahasim me pragun hapësinor të komunës nga Pjesa I (1,866 m). Moda 7,071 m trajtohet si kod i përgjithësimit, jo si përjashtues. | 5 vlera mbi 100 km (462,462–4,226,661 m), më të mëdha se vetë Kosova. Të gjitha janë nga iNaturalist: ka gjasë gabim njësie ose vlerë e pajisjes. Pasiguri më e madhe se komuna e shënuar. | 7,071 m (8,225 rreshta, 56.2% e të plotësuarave) është gjysma e diagonales së katrorit 10 × 10 km: koordinatë e përgjithësuar me qëllim. 10–100 km: 85 rreshta, 79 nga iNaturalist, ndoshta koordinata të fshehura për specie të ndjeshme. ≤ 5 m (1,068 rreshta): GPS i saktë. |
| `year` | Jo IQR ose Z global: vitet 2018–2022 kanë 58.8% të regjistrimeve, prandaj IQR shënon çdo vit para 2012 (6,180 rreshta). IQR / MAD brenda `basisOfRecord`. Kontroll kundrejt `event_start` dhe `date_precision`. Për numrin e regjistrimeve sipas vitit/muajit: rolling median ± k·MAD te `agg_time_v2.csv`. | Vit pas datës së shkarkimit (regjistrimi i fundit: 2026-08-22). Viti i digjitalizimit në vend të vitit të mbledhjes. Vit që kundërshton `event_start` ose `date_precision`. 2,268 rreshta nuk kanë vit. | 16 regjistrime para vitit 1900 (më i vjetri 1826), 15 prej tyre ekzemplarë herbariumi (`PRESERVED_SPECIMEN`). Janë dëshmi historike. Rritja pas 2018 vjen nga përpjekja e mbledhjes (eBird), jo nga ndryshim real. |
| `decimalLatitude` / `decimalLongitude` | Kontroll i kutisë së Kosovës (lat 41.85–43.27, lon 20.01–21.80). Distanca haversine deri te k = 5 fqinjët më të afërt, DBSCAN. Isolation Forest / LOF. IQR veç e veç mbi lat dhe lon nuk i gjen pikat e izoluara. | Pikë jashtë Kosovës (0 tani: lat 41.859–43.242, lon 20.021–21.763). Lat dhe lon të ndërruara. Koordinata me ≤ 2 shifra dhjetore (586 rreshta), qendër komune ose rrjeti. Pikë që nuk përputhet me `municipality`. | Pika e Ligatinave të Henc (42.579243, 21.048304) ka 12,943 rreshta (27.9%): hotspot real, por i mbipërfaqësuar. Pikat e izoluara në male ose afër kufirit janë zona pak të vizituara, jo gabime. |
| `n_issues` | Nuk detektohet statistikisht: vlerat janë diskrete 0–3 dhe Q1 = Q3 = 0, prandaj IQR do të shënonte çdo rresht me ≥ 1 flamur (1,645). Përdoret si variabël konteksti: krahasohet me `n_metoda` te flamujt. | Numri vetë nuk është matje e gabuar. Gabimi është te kolona që e ka flamurin (data, taksoni, pasiguria). 394 rreshta me ≥ 2 flamuj kanë gjasë më të madhe gabimi. | 10 rreshta me 3 flamuj janë të rrallë, por GBIF i ka pranuar. Flamujt informues, p.sh. `TAXON_MATCH_HIGHERRANK`, nuk e bëjnë regjistrimin të gabuar. |
| `regjistrime` (komuna) | Mahalanobis mbi 38 komunat (regjistrime, specie unike, specie/km², regjistrime/1,000 banorë, % e kërcënuara), pas `log1p` dhe Z-score, me prag χ² (p = 0.975). IQR mbi `log1p` si krahasim (3 komuna). | Regjistrime në komunë të gabuar ose të numëruara dy herë. Duplikatet u hoqën në v2, prandaj mbetet rasti i komunës së gabuar. | Graçanicë 14,563 regjistrime (31.35%), nga pika e Henc-it. Mamushë 1 regjistrim: komunë e vogël dhe pak e vizituar. |
| `specie_unike` (komuna) | Mahalanobis si më lart. IQR mbi `log1p` (4 komuna). | Sinonime ose emra në trajtime taksonomike të ndryshme që numërohen si specie të ndryshme (p.sh. *Lasiommata* / *Pararge* nga Pjesa I). | Më shumë: Prizren 1,243, Prishtinë 1,149. Më pak: Mamushë 1, Kllokot 13. Prizreni ka Sharrin. Komunat e vogla kanë pak specie sepse kanë pak regjistrime. |
| `specie_per_km2` (komuna) | Mahalanobis si më lart. IQR mbi `log1p` (0 komuna). | Sipërfaqe e gabuar ose e përbashkët: Mitrovicë e Jugut dhe Mitrovicë e Veriut kanë një njësi sipërfaqeje të përbashkët. | Më shumë: Junik 3.37, Ranillug 3.18. Sipërfaqe e vogël dhe vende me shumë specie. |
| `regjistrime_per_1000_banore` (komuna) | Mahalanobis si më lart. IQR mbi `log1p` (0 komuna). | Popullsia është vlerësim i ASK për 4 komuna (Leposaviq, Zveçan, Zubin Potok, Mitrovicë e Veriut), prandaj raporti është më pak i sigurt aty. | Më shumë: Graçanicë 787.79, Ranillug 392.18. Përpjekje e lartë vëzhgimi në komuna me pak banorë. |
| `specie_te_kercenuara` / `regjistrime_te_kercenuara` (komuna) | Mahalanobis si më lart, me % e regjistrimeve të kërcënuara. IQR mbi `log1p` (0 dhe 2 komuna). | Kategoria IUCN është globale, jo për Kosovën. Një identifikim i gabuar i një specieje të kërcënuar e rrit numrin. | Regjistrime: Graçanicë 365, Fushë Kosovë 81. Specie: Prishtinë 13, Prizren 12. 9 komuna nuk kanë asnjë. |

### 1.4 Vendimi dhe arsyetimi

In [5]:
report = f"""# Plani i detektimit të përjashtuesve

**Versioni:** 1.0
**Data:** 18.11.2026
**Statusi:** Plani. Metodat zbatohen 19–28 nëntor në `notebooks/04_perjashtuesit.ipynb`.

## Varësia

Input: `processed/biodiversity_features.csv` ({f['N']:,} rreshta, Pjesa I) dhe `processed/agg_municipality_v2.csv` (38 komuna). Kufijtë fizikë vijnë nga Pjesa I: lartësia {ELEV_MIN}–{ELEV_MAX:,} m (`reports/vlefshmeria.md`) dhe kutia e Kosovës (`reports/korniza_e_kualitetit.md`). Pamja e parë e kolonave është te `reports/04_kolonat_kandidate.csv`.

## Tabela e planit

{plan_table}

## Vendimi dhe arsyetimi

- **Asnjë vlerë nuk fshihet këtu.** Çdo metodë jep një flamur 0/1. Vlera largohet vetëm kur Deliza provon se është gabim (`05_menjanimi.ipynb`, 24–25 nëntor).
- **`log1p` për kolonat e anuara.** `individualCount`, `coordinateUncertaintyInMeters` dhe agregatet e komunave kanë bisht të gjatë djathtas. IQR lineare te `individualCount` shënon {f['ic_iqr_lin']:,} vlera, mbi `log1p` {f['ic_iqr_log']:,}.
- **Metodat kontekstuale para atyre globale.** Lartësia dhe numri i individëve varen nga grupi: kërpudhat jetojnë më lart se insektet, zogjtë fluturojnë në tufa. Vlera ekstreme globalisht mund të jetë normale brenda grupit.
- **`year` nuk kontrollohet me IQR globale.** Anshmëria e eBird e bën IQR të shënojë {f['yr_iqr_lin']:,} rreshta para vitit {f['yr_iqr_low']:.0f}, përfshirë ekzemplarët historikë.
- **`n_issues` nuk është kolonë për detektim.** Është numër flamujsh, jo matje. Përdoret për të parë nëse përjashtuesit kanë më shumë flamuj GBIF.
- **7,071 m është kod, jo përjashtues.** Është vlera më e shpeshtë e pasigurisë dhe tregon përgjithësim në rrjet 10 × 10 km.
- **Komunat kontrollohen me Mahalanobis.** Me 38 komuna, Isolation Forest dhe LOF nuk kanë mjaft pika. Mahalanobis i sheh variablat së bashku, p.sh. Graçanicën me shumë regjistrime, por jo me shumë specie.

## Dokumentimi i detyrës

- **Input:** `processed/biodiversity_features.csv` (`individualCount`, `elevation`, `coordinateUncertaintyInMeters`, `year`, `decimalLatitude`, `decimalLongitude`, `n_issues`) dhe `processed/agg_municipality_v2.csv`.
- **Çka u bë:** U krijua `notebooks/04_perjashtuesit.ipynb` dhe folderi `reports/figures/`. U profilizuan 13 kolona kandidate dhe për secilën u shkrua metoda, gabimi i mundshëm dhe vlera reale e rrallë.
- **Rreshta para / pas:** {f['N']:,} / {f['N']:,}. Asnjë vlerë nuk u shënua ende.
- **Pse u bë:** Që para detektimit të dihet çka kërkohet në secilën kolonë dhe që vlerat reale ekstreme të mos fshihen si gabime.
- **Output:** `reports/perjashtuesit_plani.md`, `reports/04_kolonat_kandidate.csv`, `reports/figures/`, seksioni 1 i `notebooks/04_perjashtuesit.ipynb`.
- **Commit:** `Dafina: plani i detektimit të përjashtuesve`.
- **Decision + Rationale:** shih seksionin "Vendimi dhe arsyetimi".
"""
(REPORTS_DIR / "perjashtuesit_plani.md").write_text(report, encoding="utf-8")
assert "| Kolona | Metoda | Gabim i mundshëm | Vlerë reale e rrallë |" in report
print("U ruajt:", REPORTS_DIR / "perjashtuesit_plani.md")

U ruajt: C:\Master\PVDH\reports\perjashtuesit_plani.md


### Dokumentimi i detyrës (18 nëntor)

1. **Input:** `processed/biodiversity_features.csv` (46,456 rreshta) dhe `processed/agg_municipality_v2.csv` (38 komuna).
2. **Çka u bë:** U krijua ky notebook dhe folderi `reports/figures/`. U profilizuan 13 kolona kandidate (7 në nivel rreshti, 6 agregate komunash). Për secilën u shkrua metoda, gabimi i mundshëm dhe vlera reale e rrallë.
3. **Rreshta para / pas:** 46,456 / 46,456. Asnjë vlerë nuk u shënua ose u ndryshua.
4. **Pse u bë:** Që detektimi i 19–28 nëntorit të ketë rregulla të qarta, dhe që vlerat reale ekstreme të mos ngatërrohen me gabime.
5. **Output:** `reports/perjashtuesit_plani.md` (Kolona | Metoda | Gabim i mundshëm | Vlerë reale e rrallë), `reports/04_kolonat_kandidate.csv`, `reports/figures/`.
6. **Commit:** `Dafina: plani i detektimit të përjashtuesve`.
7. **Decision + Rationale:** Asnjë fshirje, vetëm flamuj. `log1p` për kolonat e anuara, metoda kontekstuale sipas grupit, `year` pa IQR globale, `n_issues` vetëm si kontekst, 7,071 m si kod, Mahalanobis për komunat. Arsyetimi i plotë është te `reports/perjashtuesit_plani.md`.